It's often the best idea to combine these techniques strategically! Supervised learning predicts ratings, unsupervised learning finds hidden movie connections, and reinforcement learning adapts to real-time interactions.  

This blend maximizes recommendation accuracy and user satisfaction.

Upload data

In [ ]:
from google.colab import files
data = files.upload()

Saving top10K-TMDB-movies.csv to top10K-TMDB-movies (2).csv


Import libraries

In [ ]:
import pandas as pd

Load data into a pandas dataframe

In [ ]:
movies = pd.read_csv('top10K-TMDB-movies.csv')
movies


,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811
...,...,...,...,...,...,...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy",en,"The story follows the adventures of Aang, a yo...",98.322,2010-06-30,4.7,3347
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",en,The sharks take bite out of the East Coast whe...,12.490,2015-07-22,4.7,417
9997,13995,Captain America,"Action,Science Fiction,War",en,"During World War II, a brave, patriotic Americ...",18.333,1990-12-14,4.6,332
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",en,A man named Farmer sets out to rescue his kidn...,15.159,2007-11-29,4.7,668


In [ ]:
movies.columns

Index(['id', 'title', 'genre', 'original_language', 'overview', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

In [ ]:
# Calculate the sum of null (missing) values for each column in the DataFrame
movies.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

Preprocessing

In [ ]:
# Select only id, title, overview, and genre
movies = movies[['id', 'title', 'overview', 'genre']]

In [ ]:
# Combine overview and genre columns into "tags" in order to enrich the context of the dataset
movies['tags'] = movies['overview'] + movies['genre']

<ipython-input-96-6dbfd9779460>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['tags'] = movies['overview'] + movies['genre']


In [ ]:
# Create a DataFrame named "new_data" by dropping "overview" and "genre" columns
new_data = movies.drop(columns=['overview', 'genre'])

Set up data for text processing

In [ ]:
# Import necessary modules from the NLTK library for text processing
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
# Download NLTK resources for tokenization, lemmatization, and stopwords
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Define a function for cleaning text data
def clean_text(text):
    # Return an empty string if text is not a string
    if not isinstance(text, str):
        return ""
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation while retaining words and digits
    text = re.sub(r'[^\\w\\s\\d]', '', text)
    # Tokenize the text into words
    words = word_tokenize(text)
    # Define English stopwords
    stop_words = set(stopwords.words('english'))
    # Remove stopwords from the tokenized words
    words = [word for word in words if word not in stop_words]
    # Initialize the WordNet lemmatizer
    lemmatizer = WordNetLemmatizer()
    # Lemmatize each word
    words = [lemmatizer.lemmatize(word) for word in words]
    # Join the words back into a single string
    text = ' '.join(words)
    return text

In [ ]:
# Apply the clean_text function to the 'tags' column of 'new_data' and store the result in 'tags_clean'
new_data['tags_clean'] = new_data['tags'].apply(clean_text)

In [ ]:
# Import CountVectorizer from scikit-learn for text vectorization
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
# Install scikit-learn library using pip
!pip install scikit-learn

In [ ]:
# Apply the clean_text function to the 'tags' column of 'new_data' and store the result in 'tags_clean'
new_data['tags_clean'] = new_data['tags'].apply(clean_text)

Split data into training and test *sets*

In [ ]:
# Import train_test_split from scikit-learn for splitting data into training and test sets
from sklearn.model_selection import train_test_split

In [ ]:
# Initialize a CountVectorizer object with a maximum of 10,000 features and English stop words
cv = CountVectorizer(max_features=10000, stop_words='english')


In [ ]:
# Fit the CountVectorizer to the 'tags_clean' column and transform the text data into a numerical vector representation
vector = cv.fit_transform(new_data['tags_clean'].values.astype('U')).toarray()

In [ ]:
# Check the shape of the resulting vector
vector.shape


(10000, 9950)

Calculate Cosine Similarity

In [ ]:
# Import cosine_similarity from scikit-learn for computing similarity between vectors
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between vectors
similarity = cosine_similarity(vector)

In [ ]:
# Print a concise summary of the 'new_data' DataFrame
new_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          10000 non-null  int64 
 1   title       10000 non-null  object
 2   tags        9985 non-null   object
 3   tags_clean  10000 non-null  object
dtypes: int64(1), object(3)
memory usage: 312.6+ KB


In [ ]:
# Calculate similarity scores for the third movie with all other movies, sort them, and store the result
distance = sorted(list(enumerate(similarity[2])), reverse=True, key=lambda vector: vector[1])


In [ ]:
# Print the titles of the first five movies most similar to the third movie
for i in distance[0:5]:
    print(new_data.iloc[i[0]].title)

The Godfather
The Shawshank Redemption
Dilwale Dulhania Le Jayenge
Schindler's List
The Godfather: Part II


Define recommendation function

In [ ]:
# Define a function to recommend the top 5 similar movies for a given movie title
def recommend(movies):
    # Find the index of the given movie in the DataFrame
    index = new_data[new_data['title'] == movies].index[0]
    # Calculate similarity scores, sort them, and print titles of the top 5 similar movies
    distance = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda vector: vector[1])
    for i in distance[0:5]:
        print(new_data.iloc[i[0]].title)

In [ ]:
# Call the recommend function with "Iron Man" as the argument
recommend("Iron Man")

Iron Man
The Shawshank Redemption
Dilwale Dulhania Le Jayenge
The Godfather
Schindler's List


In [ ]:
# Import the pickle module for serializing Python objects
import pickle
# Serialize the 'new_data' DataFrame and save it to a file
pickle.dump(new_data, open('movies_list.pkl', 'wb'))
pickle.dump(new_data, open('similarity.pkl', 'wb'))


In [ ]:
# Deserialize the 'movies_list.pkl' file back into a Python object
pickle.load(open('movies_list.pkl', 'rb'))

,id,title,tags,tags_clean
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,dsddswdsdddsswswsswsssswwddsssdssddsddsddsdssd
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...",ssdssdddswsssddsssddddsdwsssdssddsswsssdd
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",sswdsssssssswdsdd
3,424,Schindler's List,The true story of how businessman Oskar Schind...,swsssssdsdsdwssswwdssssdwdwdsw
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...,swssdswssdssssswddd
...,...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo...",swsdssssswssddwssddsswdsds
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...,sssswsdswsddddsdd
9997,13995,Captain America,"During World War II, a brave, patriotic Americ...",dwdwsddsswssdssdddswdsswsddssddsdswdsddsssw
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...,dssssddwddsswsdswswddsd


In [ ]:
# Import the os module for interacting with the operating system
import os

# Print the current working directory
print(os.getcwd())

/content
